# SOFC Voltage Forecasting — Colab Runner (LSTM, Seq2Seq LSTM, TCN)

Notebook này train **LSTM, Seq2Seq LSTM, TCN** cho target **`V`** (điện áp SOFC) trên Colab với GPU miễn phí. Random Forest và XGBoost chạy riêng ở local (không cần GPU, xem `src/main.py`).

**Cách hoạt động:** `E:\sofc` giờ đã là git repo (`github.com/trungthanh-dev/SOFC`, public), nên notebook này **clone/pull thẳng vào Google Drive** — giống hệt cách `E:\FCF\FCF_Colab.ipynb` hoạt động. File data thô `data/raw/DataTime_export.csv` (16MB) đã được commit thẳng vào repo (khác FCF — bên đó `data_clean_power/*.parquet` bị `.gitignore` nên phải upload tay), nên **không cần upload bất kỳ file nào thủ công** — chỉ cần mount Drive rồi chạy lần lượt các cell.

**Trước khi chạy:**
1. `Runtime -> Change runtime type -> GPU (T4)`.
2. Repo là public nên để `GITHUB_TOKEN = ""` ở cell cấu hình bên dưới là chạy được ngay.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Cấu hình repo

Repo public nên để `GITHUB_TOKEN = ""` là đủ. Chỉ cần điền token nếu sau này bạn chuyển repo sang Private (GitHub -> Settings -> Developer settings -> Personal access tokens -> Generate new token, tick quyền `repo`).


In [ ]:
GITHUB_USERNAME = "trungthanh-dev"
GITHUB_REPO = "SOFC"
GITHUB_TOKEN = ""  # dán token vào đây nếu repo Private, để trống nếu Public

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/SOFC"
SRC_DIR = f"{DRIVE_PROJECT_DIR}/src"


## 3. Clone (lần đầu) hoặc Pull (các lần sau)

Cell này tự phát hiện: nếu `DRIVE_PROJECT_DIR` chưa tồn tại trên Drive thì clone, nếu đã tồn tại thì chỉ `git pull` để lấy code mới nhất từ GitHub — bao gồm cả `data/raw/DataTime_export.csv` vì file này đã nằm trong git.


In [ ]:
import os

if GITHUB_TOKEN:
    remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"
else:
    remote_url = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"

if not os.path.exists(DRIVE_PROJECT_DIR):
    print("Chưa có project trên Drive -> clone lần đầu...")
    !git clone {remote_url} "{DRIVE_PROJECT_DIR}"
else:
    print("Project đã có trên Drive -> pull code mới nhất...")
    %cd {DRIVE_PROJECT_DIR}
    !git pull

%cd {DRIVE_PROJECT_DIR}
!git log --oneline -5


## 4. Kiểm tra file cần thiết đã có chưa

Vì clone/pull thẳng từ GitHub, bước này chỉ để xác nhận không có gì bị thiếu (ví dụ do `.gitignore` lỡ loại nhầm) trước khi chạy pipeline — không cần upload thủ công.


In [ ]:
required = [
    f"{DRIVE_PROJECT_DIR}/data/raw/DataTime_export.csv",
    f"{SRC_DIR}/config.py",
    f"{SRC_DIR}/preprocessing.py",
    f"{SRC_DIR}/features.py",
    f"{SRC_DIR}/windowing.py",
    f"{SRC_DIR}/diagnostics.py",
    f"{SRC_DIR}/main_lstm.py",
    f"{SRC_DIR}/main_tcn.py",
    f"{SRC_DIR}/main_seq2seq.py",
    f"{SRC_DIR}/main_lstm_delta.py",
    f"{SRC_DIR}/main_tcn_delta.py",
    f"{SRC_DIR}/main_seq2seq_delta.py",
    f"{SRC_DIR}/main_lstm_power.py",
    f"{SRC_DIR}/main_tcn_power.py",
    f"{SRC_DIR}/main_seq2seq_power.py",
    f"{SRC_DIR}/main_lstm_power_delta.py",
    f"{SRC_DIR}/main_tcn_power_delta.py",
    f"{SRC_DIR}/main_seq2seq_power_delta.py",
    f"{SRC_DIR}/models/__init__.py",
    f"{SRC_DIR}/models/lstm.py",
    f"{SRC_DIR}/models/tcn.py",
    f"{SRC_DIR}/models/seq2seq_lstm.py",
]
missing = [p for p in required if not os.path.exists(p)]

if missing:
    print("THIẾU các file sau trên Drive (kiểm tra lại git clone/pull ở mục 3):")
    for p in missing:
        print(" -", p)
else:
    print("Đã có đủ file cần thiết. Sẵn sàng chạy tiếp.")


## 5. Cài đặt thư viện

`torch`, `pandas`, `scikit-learn` đã có sẵn trên Colab. `xgboost` không cần cho notebook này (RF/XGBoost chạy local) nhưng `models/xgboost_model.py` bị import gián tiếp khi Python quét `models/` package -- cài luôn cho chắc, không tốn thời gian đáng kể.


In [ ]:
!pip install -q xgboost


## 6. Kiểm tra GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Không có GPU -> Runtime -> Change runtime type -> GPU (T4), rồi Runtime -> Restart session và chạy lại từ đầu.")


## 7. Chạy pipeline (khuyến nghị: 1 cell chạy hết — `Runtime -> Run all` là xong)

Cell dưới đây chạy tuần tự cả 12 script (V + Power, raw + Delta-Target, LSTM/TCN/Seq2Seq) trong 1 lần bấm — không cần bấm từng cell theo đúng thứ tự nữa. Mỗi script vẫn tự lưu model/predictions/report vào `outputs/` trên Drive như trước; nếu 1 script lỗi, cell in rõ tên script lỗi rồi **tiếp tục chạy các script còn lại** (không dừng hết cả pipeline vì 1 lỗi).

Muốn train lại **đúng 1 model cụ thể** (VD sửa hyperparameter TCN rồi chỉ muốn chạy lại TCN-delta): dùng các cell riêng lẻ ở mục 7.1-7.4 bên dưới, không cần chạy lại cell gộp này.


In [ ]:
import subprocess
import time

ALL_SCRIPTS = [
    # target V (Voltage), raw-target
    "main_lstm.py", "main_tcn.py", "main_seq2seq.py",
    # target V, Delta-Target
    "main_lstm_delta.py", "main_tcn_delta.py", "main_seq2seq_delta.py",
    # target W (Power), raw-target
    "main_lstm_power.py", "main_tcn_power.py", "main_seq2seq_power.py",
    # target W, Delta-Target
    "main_lstm_power_delta.py", "main_tcn_power_delta.py", "main_seq2seq_power_delta.py",
]

SEP = "=" * 80
failed = []
t_start = time.time()
for i, script in enumerate(ALL_SCRIPTS, 1):
    print(SEP)
    print(f"[{i}/{len(ALL_SCRIPTS)}] {script}")
    print(SEP)
    t0 = time.time()
    result = subprocess.run(["python", script], cwd=SRC_DIR)
    elapsed = time.time() - t0
    if result.returncode != 0:
        print(f"!!! {script} THẤT BẠI (exit code {result.returncode}, {elapsed:.0f}s) -- tiếp tục script kế tiếp !!!")
        failed.append(script)
    else:
        print(f"--- {script} xong ({elapsed:.0f}s) ---")

print(SEP)
print(f"Tổng thời gian: {(time.time()-t_start)/60:.1f} phút")
if failed:
    print(f"CÁC SCRIPT LỖI ({len(failed)}): {failed}")
else:
    print("Tất cả 12 script chạy xong, không lỗi.")


### 7.1 Chạy riêng — target Điện áp (V), raw-target

Mỗi model là 1 cell riêng, chạy tuần tự. Mỗi script tự: load + chuẩn bị data (`features.prepare_data()`), chia train/val/test theo run_id, tạo sliding window, train, đánh giá (MAE/RMSE/R2/DTW), rồi lưu model + predictions + bảng kết quả vào `outputs/` -- vì `outputs/` nằm ngay trong `DRIVE_PROJECT_DIR` (đang đứng trên Drive), mọi thứ tự động được giữ lại kể cả khi Colab ngắt kết nối (và không nằm trong git vì đã bị `.gitignore`).


In [ ]:
%cd {SRC_DIR}
!python main_lstm.py


In [ ]:
%cd {SRC_DIR}
!python main_tcn.py


In [ ]:
%cd {SRC_DIR}
!python main_seq2seq.py


### 7.2 Chạy riêng — target Điện áp (V), Delta-Target

Kết quả raw-target ở mục 7 cho thấy LSTM/TCN thắng đậm ở h=1 nhưng xuống dốc nhanh khi horizon tăng -- dấu hiệu persistence bias (model dựa nhiều vào `V_Lag1`, xem `notes/SOFC_data_notes.md` mục 14.3/16). Delta-Target Reformulation train model trên `y(t+h) - y(t)` thay vì giá trị thô, loại bỏ việc "chép Lag1" như một lối tắt miễn phí.

Baseline raw-target (`main_lstm.py`/`main_tcn.py`/`main_seq2seq.py`, mục 7) **không bị ghi đè** -- 3 script dưới đây ghi kết quả vào `*_delta_results.csv` riêng để so sánh trực tiếp 2 phiên bản.


In [ ]:
%cd {SRC_DIR}
!python main_lstm_delta.py


In [ ]:
%cd {SRC_DIR}
!python main_tcn_delta.py


In [ ]:
%cd {SRC_DIR}
!python main_seq2seq_delta.py


### 7.3 Chạy riêng — target Công suất (W), raw-target

Mở rộng sang dự đoán `W` (Power) thay vì `V` (xem `notes/SOFC_data_notes.md` mục 25) — cùng pipeline, đổi target qua `features.prepare_data(target="W")`. `V` và `I` được giữ lại làm feature (không phải leakage, khác với lý do loại `W` khi dự đoán `V`). Kết quả RF/XGBoost-Power (local) cho thấy đây là bài toán khó hơn `V` rõ rệt (R² thấp hơn, giảm nhanh hơn theo horizon) — 3 cell dưới đây kiểm tra xem LSTM/TCN/Seq2Seq có làm tốt hơn không.


In [ ]:
%cd {SRC_DIR}
!python main_lstm_power.py


In [ ]:
%cd {SRC_DIR}
!python main_tcn_power.py


In [ ]:
%cd {SRC_DIR}
!python main_seq2seq_power.py


### 7.4 Chạy riêng — target Công suất (W), Delta-Target

RF/XGBoost-Power-delta (local, mục 25.5) vẫn thua persistence baseline ở mọi horizon — chưa xác nhận được giả thuyết "Power là nơi Delta-Target có giá trị thực tiễn nhất" (persistence của Power yếu hẳn ở horizon dài, R²≈0.008 ở h=20 — nhiều dư địa hơn cho model thật). Đây là phép thử quyết định: LSTM/TCN/Seq2Seq-delta từng thắng dần persistence theo horizon trên `V` (mục 22.3) — xem có lặp lại trên `W` không.


In [ ]:
%cd {SRC_DIR}
!python main_lstm_power_delta.py


In [ ]:
%cd {SRC_DIR}
!python main_tcn_power_delta.py


In [ ]:
%cd {SRC_DIR}
!python main_seq2seq_power_delta.py


## 8. So sánh tất cả model (Colab) — V và W, raw và delta

Đọc lại `outputs/reports/*.csv` để so sánh, không cần chạy lại cell nào ở mục 7.


In [ ]:
import pandas as pd

REPORTS_DIR = f"{DRIVE_PROJECT_DIR}/outputs/reports"

combined = []
for name, fname in [
    ("LSTM", "lstm_results.csv"), ("TCN", "tcn_results.csv"), ("Seq2Seq", "seq2seq_results.csv"),
    ("LSTM-delta", "lstm_delta_results.csv"), ("TCN-delta", "tcn_delta_results.csv"), ("Seq2Seq-delta", "seq2seq_delta_results.csv"),
    ("LSTM-power", "lstm_power_results.csv"), ("TCN-power", "tcn_power_results.csv"), ("Seq2Seq-power", "seq2seq_power_results.csv"),
    ("LSTM-power-delta", "lstm_power_delta_results.csv"), ("TCN-power-delta", "tcn_power_delta_results.csv"), ("Seq2Seq-power-delta", "seq2seq_power_delta_results.csv"),
]:
    path = f"{REPORTS_DIR}/{fname}"
    if os.path.exists(path):
        d = pd.read_csv(path)
        d.insert(0, "model", name)
        combined.append(d)
    else:
        print(f"(chưa có {fname} -- chạy cell tương ứng ở mục 7 trước)")

combined_df = pd.concat(combined, ignore_index=True) if combined else pd.DataFrame()
combined_df


## 9. Ghi chú

- Mỗi lần **sửa code ở máy local**, nhớ `git push` trước khi mở lại Colab, rồi chạy lại Cell 3 (`!git pull`) để đồng bộ -- không cần upload tay nữa.
- Kết quả (`outputs/reports/*.csv`), model (`outputs/models_saved/`), và predictions (`outputs/predictions_cache/`) đều nằm trên Drive, **không nằm trong git** (`.gitignore`) -> tải thủ công về `E:\sofc\outputs\` nếu muốn gộp chung với kết quả Random Forest / XGBoost chạy local (`src/main.py`).
- Random Forest và XGBoost KHÔNG chạy ở đây (không cần GPU, chạy local nhanh hơn nhiều qua `python src/main.py`).
- Hyperparameter (`hidden_size`, `num_channels`, `epochs`, `patience`...) trong `main_lstm.py`/`main_tcn.py`/`main_seq2seq.py` là điểm khởi đầu, chưa tune riêng theo dataset SOFC -- nếu train loss không giảm hoặc R2 âm, sửa trực tiếp trong script đó ở local, `git push`, rồi `git pull` lại trên Colab và chạy lại cell tương ứng.
- Muốn train lại 1 model cụ thể: chỉ cần chạy lại đúng cell của model đó ở mục 7/7b, không cần chạy lại toàn bộ notebook.

- Kết quả target Power (`*_power_results.csv`, `*_power_delta_results.csv`) tải về `E:\sofc\outputs\` giống hệt cách làm với target Voltage (mục 20 trong notes).